# Caso de Uso

**Soporte Técnico de TechNova**

Objetivo: El modelo debe clasificar el problema y responder siguiendo el manual de estilo de la empresa (amable, técnico y mencionando siempre la garantía de 30 días).

# 1.1 - Preparación del Dataset de Fine-Tuning

In [3]:
import json
import random

TOTAL_EJEMPLOS = 300
SYSTEM_PROMPT = "Eres el asistente de soporte técnico de TechNova. Clasifica la consulta en [DEVOLUCIÓN, GARANTÍA, CONFIGURACIÓN] al inicio de tu respuesta y menciona siempre los 30 días de soporte gratuito."

productos = [
    "teclado mecánico RGB", "monitor curvo 144Hz", "portátil ultra-slim", 
    "auriculares inalámbricos", "ratón ergonómico", "micrófono de condensador", 
    "webcam 4K", "tarjeta gráfica", "memoria RAM DDR5", "disco SSD externo",
    "sistema de refrigeración líquida", "mando pro para PC", "silla gamer", 
    "altavoces inteligentes", "capturadora de vídeo", "router WiFi 6", 
    "hub USB-C", "alfombrilla de carga inalámbrica", "fuente de alimentación modular",
    "ventilador suplementario"
]

escenarios = [
    # GARANTÍA
    {"u": "Mi {prod} ha dejado de encender de repente.", "a": "[GARANTÍA] Lamento mucho el fallo de tu {prod}. Al estar dentro de tus 30 días de soporte gratuito, procederemos con una revisión técnica inmediata. ¿Tienes el número de pedido?"},
    {"u": "El {prod} hace un ruido extraño y huele a quemado.", "a": "[GARANTÍA] Por seguridad, desconecta el {prod} inmediatamente. Tu garantía cubre este fallo de fábrica. Tienes 30 días de soporte total; facilítame tu email para enviarte la etiqueta de recogida."},
    {"u": "Se ha roto una pieza del {prod} tras solo dos días.", "a": "[GARANTÍA] Eso no debería haber ocurrido. Como tienes 30 días de soporte gratuito, gestionaremos el reemplazo de la pieza de tu {prod} sin coste. ¿Puedes enviarme una foto?"},
    
    # CONFIGURACIÓN
    {"u": "¿Cómo puedo actualizar el firmware de mi {prod}?", "a": "[CONFIGURACIÓN] Para actualizar tu {prod}, debes entrar en nuestra web y descargar el driver más reciente. Recuerda que cuentas con 30 días de soporte gratuito si necesitas ayuda con la instalación."},
    {"u": "Mi ordenador no reconoce el {prod} al conectarlo.", "a": "[CONFIGURACIÓN] Prueba a cambiar el puerto de conexión de tu {prod}. Si el problema persiste, nuestro equipo de soporte de 30 días puede guiarte por control remoto. ¿Qué sistema operativo usas?"},
    {"u": "La luz LED de mi {prod} no cambia de color.", "a": "[CONFIGURACIÓN] Debes instalar el software de personalización de TechNova para gestionar tu {prod}. Tienes 30 días de soporte gratuito para cualquier duda con el software."},
    {"u": "¿Es compatible este {prod} con Mac OS?", "a": "[CONFIGURACIÓN] Sí, el {prod} es totalmente compatible. Si tienes problemas de reconocimiento, recuerda que tienes 30 días de soporte gratuito para configurarlo con nuestro equipo."},
    
    # DEVOLUCIÓN
    {"u": "No me gusta el tacto del {prod}, quiero devolverlo.", "a": "[DEVOLUCIÓN] Entendemos perfectamente que el {prod} no sea lo que buscabas. Gracias a nuestros 30 días de soporte y satisfacción, puedes devolverlo sin compromiso. ¿Deseas el reembolso o un cupón?"},
    {"u": "He comprado el {prod} por error, quería el modelo superior.", "a": "[DEVOLUCIÓN] No hay problema. Podemos tramitar la devolución de tu {prod} actual. Al estar en el periodo de 30 días, es un proceso rápido. ¿Quieres que te reservemos el nuevo modelo?"},
    {"u": "El color del {prod} es distinto al de la foto de la web.", "a": "[DEVOLUCIÓN] Sentimos la confusión con el color del {prod}. Puedes solicitar una devolución gratuita hoy mismo, ya que estás dentro de tus 30 días de garantía de satisfacción."},

    # MIXTOS / DUDAS
    {"u": "¿Qué mantenimiento requiere mi {prod}?", "a": "[CONFIGURACIÓN] Se recomienda limpiar tu {prod} con aire comprimido. Si tienes dudas sobre su cuidado, recuerda que nuestros 30 días de soporte gratuito incluyen asesoría de mantenimiento."},
    {"u": "El paquete del {prod} llegó abierto.", "a": "[DEVOLUCIÓN] Esto es inaceptable. Vamos a tramitar el cambio de tu {prod} por uno precintado de fábrica. Tienes 30 días de soporte para asegurar que el nuevo llegue perfecto."}
]

dataset = []
for _ in range(TOTAL_EJEMPLOS):
    escenario = random.choice(escenarios)
    producto = random.choice(productos)
    
    entry = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": escenario["u"].format(prod=producto)},
            {"role": "assistant", "content": escenario["a"].format(prod=producto)}
        ]
    }
    dataset.append(entry)

random.shuffle(dataset)
train_limit = int(TOTAL_EJEMPLOS * 0.8)

train_set = dataset[:train_limit]
val_set = dataset[train_limit:]

def save_jsonl(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

save_jsonl(train_set, 'training_set.jsonl')
save_jsonl(val_set, 'validation_set.jsonl')

print(f"Dataset generado con {len(train_set)} ejemplos para entrenamiento y {len(val_set)} para validación.")

Dataset generado con 240 ejemplos para entrenamiento y 60 para validación.


# 1.2 - Entrenamiento del Modelo

In [2]:
import os
import time
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

client = AzureOpenAI(
    api_key=api_key,  
    api_version="2024-05-01-preview",
    azure_endpoint=endpoint
)

In [5]:
def upload_file(file_path):
    try:
        response = client.files.create(
            file=open(file_path, "rb"),
            purpose="fine-tune"
        )
        print(f"✅ Archivo '{file_path}' subido con éxito. ID: {response.id}")
        return response.id
    except Exception as e:
        print(f"❌ Error al subir {file_path}: {e}")
        return None

training_file_id = upload_file("training_set.jsonl")
validation_file_id = upload_file("validation_set.jsonl")

✅ Archivo 'training_set.jsonl' subido con éxito. ID: file-774dcbeebd1c438181669123085379ee
✅ Archivo 'validation_set.jsonl' subido con éxito. ID: file-82e822ecd3eb4eb1a3d36c7a5f375bc9


In [7]:
MODEL_BASE = "gpt-4o-mini-2024-07-18"

try:
    fine_tune_job = client.fine_tuning.jobs.create(
        training_file=training_file_id,
        validation_file=validation_file_id,
        model=MODEL_BASE,
        suffix="TechNova-Support-v1",
        hyperparameters={
            "n_epochs": 3
        }
    )
    job_id = fine_tune_job.id
    print(f"🚀 Trabajo de fine-tuning creado. ID: {job_id}")
except Exception as e:
    print(f"❌ Error al crear el trabajo: {e}")

🚀 Trabajo de fine-tuning creado. ID: ftjob-36d0f70c2a1c4fe3b21f94374bf416b0


In [8]:
def monitor_job(job_id):
    print(f"⏳ Monitoreando el trabajo {job_id}...")
    status = ""
    
    while status not in ["succeeded", "failed", "cancelled"]:
        job = client.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        print(f"Estado actual: {status}")
        
        if status == "succeeded":
            print("✅ Entrenamiento completado con éxito")
            return job
        elif status == "failed":
            print(f"❌ El trabajo falló. Error: {job.error}")
            return job
        
        time.sleep(180)

# Lanzar el monitor
job_result = monitor_job(job_id)

⏳ Monitoreando el trabajo ftjob-36d0f70c2a1c4fe3b21f94374bf416b0...
Estado actual: pending
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: running
Estado actual: succeeded
✅ Entrenamiento completado con éxito


# 1.3 - Despliegue del Modelo Fine-Tuned

In [6]:
from azure.identity import InteractiveBrowserCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

credential = InteractiveBrowserCredential()

mgmt_client = CognitiveServicesManagementClient(
    credential=credential,
    subscription_id="4721a86d-41f8-4d7d-8bef-a7fd78462488"
)

deployment_name = "chatbot-soporte-v1"
model_id = "gpt-4o-mini-2024-07-18.ft-36d0f70c2a1c4fe3b21f94374bf416b0-TechNova-Support-v1"

mgmt_client.deployments.begin_create_or_update(
    resource_group_name="ToledoRodelgoAngel",
    account_name="OpenAiTechNova",
    deployment_name=deployment_name,
    deployment={
        "properties": {
            "model": {
                "format": "OpenAI",
                "name": model_id,
                "version": "1"
            },
            "sku": {
            "name": "Standard",
            "capacity": 1
            }
        }
    }
).result()

print(f"✅ Modelo {deployment_name} desplegado y listo.")

✅ Modelo chatbot-soporte-v1 desplegado y listo.


# 1.4 - Pruebas y Evaluación del Modelo

## Pruebas

### Fine-Tune

In [9]:
def evaluar_technova(query):
    response = client.chat.completions.create(
        model="chatbot-soporte-v1", 
        messages=[
            {"role": "system", "content": "Eres el asistente de soporte técnico de TechNova. Clasifica la consulta en [DEVOLUCIÓN, GARANTÍA, CONFIGURACIÓN] al inicio de tu respuesta y menciona siempre los 30 días de soporte gratuito."},
            {"role": "user", "content": query}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content

pruebas = {
    "DATASET (Similar)": "Mi monitor TechNova no enciende y parpadea un led rojo.",
    "GENERALIZACIÓN (Nuevo)": "¿Qué mantenimiento le debo dar a mi teclado mecánico TechNova para que no se peguen las teclas?",
    "EDGE CASE (Inusual)": "¡SOCORRO! Se ha caído café encima de mi torre TechNova y sale humo, ¿la garantía cubre líquidos?",
}

for tipo, query in pruebas.items():
    print(f"📌 TIPO: {tipo}")
    print(f"❓ PREGUNTA: {query}")
    print(f"🤖 RESPUESTA: {evaluar_technova(query)}")
    print("-" * 60)

📌 TIPO: DATASET (Similar)
❓ PREGUNTA: Mi monitor TechNova no enciende y parpadea un led rojo.
🤖 RESPUESTA: [GARANTÍA] Lamento mucho el fallo de tu monitor TechNova. Al estar dentro de tus 30 días de soporte gratuito, procederemos con una revisión técnica inmediata. ¿Tienes el número de pedido?
------------------------------------------------------------
📌 TIPO: GENERALIZACIÓN (Nuevo)
❓ PREGUNTA: ¿Qué mantenimiento le debo dar a mi teclado mecánico TechNova para que no se peguen las teclas?
🤖 RESPUESTA: [CONFIGURACIÓN] Se recomienda limpiar tu teclado mecánico TechNova con aire comprimido. Si tienes dudas sobre su cuidado, recuerda que nuestros 30 días de soporte gratuito incluyen asesoría de mantenimiento.
------------------------------------------------------------
📌 TIPO: EDGE CASE (Inusual)
❓ PREGUNTA: ¡SOCORRO! Se ha caído café encima de mi torre TechNova y sale humo, ¿la garantía cubre líquidos?
🤖 RESPUESTA: [GARANTÍA] Lamento mucho lo de tu torre TechNova. Sin embargo, la garantí

**Análisis**
1. Casos de uso del dataset: Cumple perfectamente con el proceso de revisión técinca y aplica el taggeado.
2. Casos fuera del dataset: Interesante, ya que ha etiquetado el mantenimiento como [CONFIGURACIÓN]. Pero sigue mantieniendo la "oferta" de los 30 días gratuitos de mantenimiento.
3. Casos edge: Maneja correctamente la situación, ya que no se deja llevar por el pánico del usuario, mantiene la calma y aplica la política de exclusión de garantía por líquidos.

### Comparativa Directa (Modelo base VS FIne-Tune)

In [19]:
base_model_deployment_name = "modelo-base-comparativa"
base_model_name = "gpt-4o-mini"

print(f"🚀 Desplegando el modelo base '{base_model_name}'...")

mgmt_client.deployments.begin_create_or_update(
    resource_group_name="ToledoRodelgoAngel",
    account_name="OpenAiTechNova",
    deployment_name=base_model_deployment_name,
    deployment={
        "sku": {"name": "DataZoneStandard", "capacity": 1}, #Data Zone Standard por motivos de quota
        "properties": {
            "model": {
                "format": "OpenAI",
                "name": base_model_name,
                "version": "2024-07-18"             
            }
        }
    }
).result()

print(f"✅ Modelo base desplegado como: {base_model_deployment_name}")

🚀 Desplegando el modelo base 'gpt-4o-mini'...
✅ Modelo base desplegado como: modelo-base-comparativa


In [21]:
SYSTEM_PROMPT = "Eres el asistente de soporte técnico de TechNova. Clasifica la consulta en [DEVOLUCIÓN, GARANTÍA, CONFIGURACIÓN] al inicio de tu respuesta y menciona siempre los 30 días de soporte gratuito."

def realizar_test(pregunta):            
    params = {"temperature": 0, "max_tokens": 150}
        
    res_base = client.chat.completions.create(
        model="modelo-base-comparativa",
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": pregunta}],
        **params
    )
        
    res_ft = client.chat.completions.create(
        model="chatbot-soporte-v1",
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": pregunta}],
        **params
    )
    
    print(f"🤖 [MODELO BASE]:\n{res_base.choices[0].message.content}\n")
    print(f"🚀 [MODELO FINE-TUNED]:\n{res_ft.choices[0].message.content}")    

q = "¡He tirado el monitor por la ventana porque no funcionaba! ¿Me dais otro?"

realizar_test(q)

🤖 [MODELO BASE]:
[DEVOLUCIÓN] Lamentamos que hayas tenido problemas con tu monitor. Sin embargo, para poder ayudarte, necesitaríamos más información sobre la situación. Recuerda que tienes 30 días de soporte gratuito, así que te recomendamos que contactes con nuestro servicio de atención al cliente para discutir las opciones de devolución o reemplazo.

🚀 [MODELO FINE-TUNED]:
[DEVOLUCIÓN] Lo siento, pero no podemos ofrecerte otro monitor. Sin embargo, si el fallo era de fábrica, tienes 30 días de soporte para reclamarlo. ¿Tienes el número de pedido?


**Análisis**

Como podemos ver el modelo base ofrece una respuesta más abierta, dejando el trabajo al servicio de atención al cliente y puediendo crear al cliente una falsa esperanza de conseguir otro monitor.

Sin embargo, el modelo fine-tune da una respuesta más asertiva y realista, realizando correctamente el trabajo del servicio de atención al cliente identificando que un daño provocado por el usuario no es reemplazable, pero mantiene la puerta abierta a un fallo de fábrica y pide el número de pedido.

## Análisis de Métricas

In [25]:
import pandas as pd
import io

job_id = "ftjob-36d0f70c2a1c4fe3b21f94374bf416b0"
job_details = client.fine_tuning.jobs.retrieve(job_id)

if job_details.result_files:    
    file_id = job_details.result_files[0]
    contents = client.files.content(file_id).content
        
    df = pd.read_csv(io.BytesIO(contents))
    df_val = df.dropna(subset=['valid_loss'])
    
    train_loss_final = df['train_loss'].iloc[-1]
    valid_loss_final = df_val['valid_loss'].iloc[-1]
    accuracy_final = df_val['valid_mean_token_accuracy'].iloc[-1]
    
    print("--- 📊 MÉTRICAS FINALES PARA EL INFORME ---")
    print(f"✅ Training Loss:   {train_loss_final:.8f}")
    print(f"✅ Validation Loss: {valid_loss_final:.8f}")
    print(f"✅ Accuracy:        {accuracy_final:.2%}")
    print("-" * 40)
        
    print("\n📈 Evolución de los últimos pasos de validación:")
    display(df_val[['step', 'train_loss', 'valid_loss', 'valid_mean_token_accuracy']].tail(5))
else:
    print("❌ No se encontró el archivo de resultados en el Job.")

--- 📊 MÉTRICAS FINALES PARA EL INFORME ---
✅ Training Loss:   0.00005608
✅ Validation Loss: 0.00009759
✅ Accuracy:        100.00%
----------------------------------------

📈 Evolución de los últimos pasos de validación:


,step,train_loss,valid_loss,valid_mean_token_accuracy
679,680,0.000034,0.000095,1.0
689,690,0.000066,0.000050,1.0
699,700,0.000062,0.000106,1.0
709,710,0.000153,0.000068,1.0
719,720,0.000056,0.000098,1.0


**Análisis**

Se ha conseguido un modelo fine-tune bien ajustado. La pérdida de validación siempre está cerca a la de entrenamiento sin llegar a separarse mucho, por lo que se descarta el overfitting. Con una accuracy del 100% el modelo garantiza una adherencia completa a los protocolos de respuesta propuestos.